In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training will be on {DEVICE}")

Training will be on cuda


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/AI/GMAv3/esolangs_messages.csv')

print(f"{df.shape[0]} entries.")
df_shown = df.drop('Author', axis=1)
df_shown.head()

31062 entries.


,Content,NumericAuthor
0,"I was right, I got +2",2
1,the rust is a bit rough but I thought it was j...,2
2,just realized that my entry makes absolutely n...,0
3,"my entry outputs: (byte index, gap in bytes)",0
4,but the gaps between characters will be differ...,0


In [ ]:
author_dict = {}
uniques = df['NumericAuthor'].unique()
uniques = sorted(uniques)

for i in uniques:
    author = df[df['NumericAuthor'] == i]['Author'].iloc[0]
    author_dict[int(i)] = author

author_dict_censored = {i:f"{s[0]}{(len(s) - 2) * '*'}{s[-1]}" for i, s in author_dict.items()}
print(author_dict_censored)

{0: 'k****r', 1: 'k****n', 2: 'l*****y', 3: 'o******0', 4: 'p*****_', 5: 'r**a'}


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(df['NumericAuthor']), y=df['NumericAuthor'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

In [ ]:
from keras.preprocessing.sequence import pad_sequences

X = df['Content']
Y = df['NumericAuthor']

X = X.to_list()
Y = Y.to_numpy()

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoConfig
import torch

model_name = 'roberta-base'

huggingface_tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
from sklearn.model_selection import train_test_split
X = huggingface_tokenizer(X, padding=True, truncation=False, return_tensors="pt")

In [ ]:
input_ids = X['input_ids']
attention_mask = X['attention_mask']

input_train, input_test, attention_train, attention_test, Y_train, Y_test = train_test_split(input_ids, attention_mask, Y, test_size=0.2, random_state=42)

input_train = input_train.to(DEVICE)
attention_train = attention_train.to(DEVICE)
Y_train = torch.tensor(Y_train, dtype=torch.long).to(DEVICE)

input_test = input_test.to(DEVICE)
attention_test = attention_test.to(DEVICE)
Y_test = torch.tensor(Y_test, dtype=torch.long).to(DEVICE)

In [ ]:
print(f"{input_train.shape=}")
print(f"{attention_train.shape=}")
print(f"{Y_train.shape=}")
print(f"{input_test.shape=}")
print(f"{attention_test.shape=}")
print(f"{Y_test.shape=}")

input_train.shape=torch.Size([24849, 54])
attention_train.shape=torch.Size([24849, 54])
Y_train.shape=torch.Size([24849])
input_test.shape=torch.Size([6213, 54])
attention_test.shape=torch.Size([6213, 54])
Y_test.shape=torch.Size([6213])


In [ ]:
BATCH_SIZE = 64

X_train = TensorDataset(input_train, attention_train, Y_train)
X_test = TensorDataset(input_test, attention_test, Y_test)

train_loader = DataLoader(X_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(X_test, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
print(f"{len(train_loader)=}")
print(f"{len(test_loader)=}")

len(train_loader)=389
len(test_loader)=98


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class GMAv3(nn.Module):
    def __init__(self, num_classes=6):
        super(GMAv3, self).__init__()
        self.huggingface = AutoModel.from_pretrained(model_name)
        self.huggingface_config = AutoConfig.from_pretrained(model_name)
        self.fc = nn.Linear(self.huggingface_config.hidden_size, num_classes)

    def forward(self, x, att_masks):
        x = self.huggingface(x, att_masks)
        x = x.last_hidden_state[:, 0, :]
        x = self.fc(x)
        return x

In [ ]:
SAVE_CHECKPOINTS = True
NUM_CLASSES = len(author_dict)

model = GMAv3(num_classes=NUM_CLASSES).to(DEVICE)
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model has {param_count} parameters")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model has 124650246 parameters


In [ ]:
from tqdm import tqdm
from datetime import datetime, timedelta
from pathlib import Path
import json

if SAVE_CHECKPOINTS:
    now = datetime.now() + timedelta(hours=3)
    foldername = now.strftime("%Y-%m-%d_%H-%M-%S")
    foldername = "./drive/MyDrive/AI/GMAv3/models/" + foldername
    print(f'Models will be saved to "{foldername}"')
    Path(foldername).mkdir(parents=True, exist_ok=False)
    with open(f'{foldername}/author_dict.json', 'w') as f:
        json.dump(author_dict, f)

# Training
EPOCHS = 10
LEARNING_RATE = 1e-5


optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

for epoch in range(EPOCHS):
    train_loop = tqdm(train_loader, desc="Training", leave=True)
    model.train()
    for batch_idx, (data, att_weights, target) in enumerate(train_loop):
        optimizer.zero_grad()
        output = model(data, att_weights)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        train_accuracy = (output.argmax(dim=1) == target).float().mean()
        train_accuracy = round(train_accuracy.item(), 4)
        train_loop.set_postfix(loss=loss.item(), accuracy=train_accuracy)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for data, att_weights, target in test_loader:
            output = model(data, att_weights)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

            val_loss += criterion(output, target).item()
    val_loss /= len(test_loader)

    test_accuracy = 100 * correct / total
    test_accuracy = round(test_accuracy, 2)
    print(f"Epoch {epoch+1}/{EPOCHS}, Test Accuracy: {test_accuracy:.2f}%\tValidation Loss: {val_loss:.2f}")
    if SAVE_CHECKPOINTS:
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss,
        }
        torch.save(checkpoint, f'{foldername}/esolangs_model_epoch_{epoch + 1}_accuracy_{test_accuracy:.2f}_val_loss_{val_loss:.2f}.pth')

yn = input("Save model? (y/n): ")
if yn == 'y' or yn == 'Y':
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }
    model_name = input("Model name: ")

    foldername = "./drive/MyDrive/AI/GMAv3/models/" + model_name
    print(f'Model will be saved to "{foldername}"')
    Path(foldername).mkdir(parents=True, exist_ok=False)
    with open(f'{foldername}/author_dict.json', 'w') as f:
        json.dump(author_dict, f)
    torch.save(checkpoint, f'{foldername}/{model_name}.pth')

Models will be saved to "./drive/MyDrive/AI/GMAv3/models/2025-04-25_00-07-26"


Training: 100%|██████████| 389/389 [01:37<00:00,  3.99it/s, accuracy=0.706, loss=0.719]


Epoch 1/10, Test Accuracy: 59.86%	Validation Loss: 1.04


Training: 100%|██████████| 389/389 [01:37<00:00,  4.00it/s, accuracy=0.588, loss=0.973]


Epoch 2/10, Test Accuracy: 59.02%	Validation Loss: 0.97


...

Epoch 8/10, Test Accuracy: 69.21%	Validation Loss: 1.09


Training: 100%|██████████| 389/389 [01:37<00:00,  3.99it/s, accuracy=0.824, loss=0.311]


Epoch 9/10, Test Accuracy: 68.47%	Validation Loss: 1.12


Training: 100%|██████████| 389/389 [01:37<00:00,  3.99it/s, accuracy=0.824, loss=0.495]


Epoch 10/10, Test Accuracy: 70.59%	Validation Loss: 1.26
Save model? (y/n): y
Model name: gma_v3_base
Model will be saved to "./drive/MyDrive/AI/GMAv3/models/gma_v3_base"
